# Análisis de resultados — Comparativa Claude vs Ollama

Cuaderno de apoyo al **capítulo de resultados** del TFM. Lee los CSV que produce
`evaluation/run_eval.py` (en `evaluation/results/`) y genera las tablas y gráficas
comparativas entre proveedores LLM.

**Requisito previo:** haber ejecutado al menos una evaluación, p. ej.

```bash
python evaluation/run_eval.py --providers claude ollama
```

**Dependencias:** `pip install -e ".[notebook]"` (jupyter, pandas, matplotlib).

Las métricas y su interpretación:

| Métrica | Mejor es | Qué mide |
|---|---|---|
| `ioc_recall` | ↑ alto | cobertura de IOCs esperados |
| `hallucination_rate` | ↓ bajo | % de afirmaciones sin respaldo verificable |
| `judge_media` | ↑ alto | calidad del informe según el LLM-as-judge (1–5) |
| `cost_usd` | ↓ bajo | coste por informe (Ollama = 0) |
| `latency_s` | ↓ bajo | latencia del pipeline |

In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Paleta categórica segura para daltonismo (Okabe-Ito). Color FIJO por proveedor
# (por entidad, no por ranking): así el color no cambia entre gráficas.
PROVIDER_COLORS = {"claude": "#0072B2", "ollama": "#E69F00"}
_FALLBACK = ["#009E73", "#CC79A7", "#56B4E9", "#D55E00", "#F0E442"]


def color_for(provider: str) -> str:
    if provider not in PROVIDER_COLORS:
        PROVIDER_COLORS[provider] = _FALLBACK[len(PROVIDER_COLORS) % len(_FALLBACK)]
    return PROVIDER_COLORS[provider]


# Estilo limpio: sin marco superior/derecho, rejilla tenue solo en Y.
plt.rcParams.update({
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.axisbelow": True,
    "figure.dpi": 110,
    "font.size": 11,
})

RESULTS_DIR = Path("..") / "evaluation" / "results"
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

ModuleNotFoundError: No module named 'matplotlib'

## 1. Cargar los resultados

Se combina la **última** ejecución de cada evaluación. Cambia `latest_only=False`
para agregar todas las ejecuciones históricas.

In [ ]:
def load_results(latest_only: bool = True) -> pd.DataFrame:
    csvs = sorted(RESULTS_DIR.glob("eval-*.csv"))
    if not csvs:
        print(
            "No hay resultados en", RESULTS_DIR.resolve(),
            "\nEjecuta primero:  python evaluation/run_eval.py --providers claude ollama",
        )
        return pd.DataFrame()
    files = [csvs[-1]] if latest_only else csvs
    df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)
    # Descartar filas de casos que fallaron (columna 'error' presente y no nula).
    if "error" in df.columns:
        df = df[df["error"].isna()].drop(columns=["error"])
    print(f"Cargadas {len(df)} filas de {len(files)} fichero(s). Proveedores:",
          sorted(df["provider"].unique()) if not df.empty else "—")
    return df


df = load_results(latest_only=True)
df

## 2. Tabla agregada por proveedor

Media de cada métrica por proveedor: la síntesis que va en el cuerpo de la memoria.

In [ ]:
METRICS = ["ioc_recall", "ioc_precision", "hallucination_rate",
           "judge_media", "cost_usd", "latency_s", "tool_calls"]

if not df.empty:
    present = [m for m in METRICS if m in df.columns]
    agg = df.groupby("provider")[present].mean().round(3)
    display(agg)
else:
    agg = pd.DataFrame()

## 3. Gráficas comparativas

Una barra por proveedor y métrica, con etiqueta de valor directa. Cada métrica en su
propio eje (nunca dos escalas en un mismo gráfico).

In [ ]:
def bar_by_provider(agg: pd.DataFrame, column: str, title: str, ylabel: str, fname: str):
    if agg.empty or column not in agg.columns:
        print(f"(sin datos para '{column}')")
        return
    data = agg[column]
    providers = list(data.index)
    colors = [color_for(p) for p in providers]

    fig, ax = plt.subplots(figsize=(5, 3.4))
    bars = ax.bar(providers, data.values, color=colors, width=0.55, zorder=3)
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=10)
    ax.set_title(title, fontsize=12, loc="left")
    ax.set_ylabel(ylabel)
    ax.set_ylim(0, max(data.values) * 1.18 if data.max() > 0 else 1)
    ax.grid(axis="x", visible=False)
    fig.tight_layout()
    fig.savefig(FIG_DIR / fname, bbox_inches="tight")
    plt.show()


bar_by_provider(agg, "ioc_recall", "Cobertura de IOCs (recall) — mayor es mejor",
                "recall", "recall.png")
bar_by_provider(agg, "hallucination_rate", "Tasa de alucinación — menor es mejor",
                "proporción", "hallucination.png")
bar_by_provider(agg, "judge_media", "Calidad (LLM-as-judge, 1–5) — mayor es mejor",
                "puntuación media", "judge.png")

### 3.1 Coste y latencia

Dos ejes separados (escalas distintas). El coste de Ollama es 0 por definición: el
contraste ilustra el trade-off coste/privacidad frente a calidad.

In [ ]:
if not agg.empty:
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
    for ax, col, title, ylabel in [
        (axes[0], "cost_usd", "Coste por informe (USD)", "USD"),
        (axes[1], "latency_s", "Latencia (s)", "segundos"),
    ]:
        if col not in agg.columns:
            continue
        providers = list(agg.index)
        bars = ax.bar(providers, agg[col].values,
                      color=[color_for(p) for p in providers], width=0.55, zorder=3)
        ax.bar_label(bars, fmt="%.4f" if col == "cost_usd" else "%.1f", padding=3, fontsize=10)
        ax.set_title(title, fontsize=12, loc="left")
        ax.set_ylabel(ylabel)
        ax.grid(axis="x", visible=False)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "coste_latencia.png", bbox_inches="tight")
    plt.show()

## 4. Detalle por caso

Alucinación caso a caso: revela si un proveedor falla de forma sistemática o solo en
objetivos concretos. Barras agrupadas por caso, color fijo por proveedor.

In [ ]:
def grouped_by_case(df: pd.DataFrame, column: str, title: str, ylabel: str, fname: str):
    if df.empty or column not in df.columns:
        print(f"(sin datos para '{column}')")
        return
    pivot = df.pivot_table(index="case", columns="provider", values=column, aggfunc="mean")
    cases = list(pivot.index)
    providers = list(pivot.columns)
    x = range(len(cases))
    n = len(providers)
    w = 0.8 / max(n, 1)

    fig, ax = plt.subplots(figsize=(max(6, len(cases) * 1.4), 3.6))
    for i, p in enumerate(providers):
        offs = [xi + (i - (n - 1) / 2) * w for xi in x]
        ax.bar(offs, pivot[p].values, width=w, label=p, color=color_for(p), zorder=3)
    ax.set_xticks(list(x))
    ax.set_xticklabels(cases, rotation=20, ha="right")
    ax.set_title(title, fontsize=12, loc="left")
    ax.set_ylabel(ylabel)
    ax.grid(axis="x", visible=False)
    ax.legend(title="proveedor", frameon=False)  # leyenda: identidad no solo por color
    fig.tight_layout()
    fig.savefig(FIG_DIR / fname, bbox_inches="tight")
    plt.show()


grouped_by_case(df, "hallucination_rate", "Tasa de alucinación por caso",
                "proporción", "hallucination_por_caso.png")
grouped_by_case(df, "ioc_recall", "Cobertura de IOCs por caso",
                "recall", "recall_por_caso.png")

## 5. Notas para la memoria

- Las figuras se guardan en `notebooks/figures/` listas para insertar en el TFM.
- Interpreta siempre las métricas en conjunto: un proveedor con `hallucination_rate`
  baja pero `ioc_recall` bajo puede ser *conservador* (dice poco pero fiable); el
  contrario puede ser *exhaustivo pero poco fiable*.
- Recuerda el trade-off del experimento: Claude (calidad, coste > 0, dato sale a la
  nube) vs Ollama (gratis, local/privado, normalmente menor calidad y más lento).
- Los datos OSINT son públicos y **vivos**: documenta la fecha de ejecución de cada
  evaluación, porque los resultados pueden variar con el tiempo.